# PROTOCOL

## Prepare project folders

In [1]:
# This notebook is running locally in VS Code, so Google Colab's drive.mount is not needed.

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

directories = [
    PROJECT_DIR / "data",
    PROJECT_DIR / "configs",
    PROJECT_DIR / "checkpoints",
    PROJECT_DIR / "predictions",
    PROJECT_DIR / "results",
    PROJECT_DIR / "logs",
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print(PROJECT_DIR)

d:\Code\unsia\enexre


## Install Libray

In [3]:
!pip install -q transformers datasets evaluate seqeval accelerate

In [4]:
import sys
import torch
import transformers
import datasets
import platform
import json

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

print(json.dumps(environment, indent=2))

with open(PROJECT_DIR / "logs" / "environment.json", "w") as file:
    json.dump(environment, file, indent=2)

{
  "python": "3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]",
  "platform": "Windows-10-10.0.19045-SP0",
  "torch": "2.12.1+cpu",
  "transformers": "5.12.1",
  "datasets": "5.0.0",
  "cuda_available": false,
  "gpu": null
}


## BC5CDR
Pembagian resmi
- Training     : 500 dokumen
- Development  : 500 dokumen
- Test         : 500 dokumen

# Eksplorasi Dataset

In [5]:
!pip install -q datasets pandas matplotlib

In [6]:
from datasets import load_dataset
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
!pip install -q -U datasets

In [8]:
from datasets import load_dataset

base_url = (
    "https://huggingface.co/datasets/"
    "tner/bc5cdr/resolve/main/dataset"
)

dataset = load_dataset(
    "json",
    data_files={
        "train": f"{base_url}/train.json",
        "validation": f"{base_url}/valid.json",
        "test": f"{base_url}/test.json",
    }
)

dataset

train.json:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

valid.json:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5228
    })
    validation: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5330
    })
    test: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5865
    })
})

In [ ]:
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5228
    })
    validation: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5330
    })
    test: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5865
    })
})
['tags', 'tokens']
{'tags': [1, 0, 0, 0, 0, 0, 1, 0], 'tokens': ['Naloxone', 'reverses', 'the', 'antihypertensive', 'effect', 'of', 'clonidine', '.']}


## Unduh BC5CDR

In [16]:
from urllib.request import urlretrieve

DATA_DIR = PROJECT_DIR / "data" / "bc5cdr"
DATA_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "train.txt": "https://ftp.ncbi.nlm.nih.gov/pub/lu/BC5CDR/CDR_TrainingSet.PubTator.txt",
    "dev.txt": "https://ftp.ncbi.nlm.nih.gov/pub/lu/BC5CDR/CDR_DevelopmentSet.PubTator.txt",
    "test.txt": "https://ftp.ncbi.nlm.nih.gov/pub/lu/BC5CDR/CDR_TestSet.PubTator.txt",
}

def show_progress(filename):
    def report(block_count, block_size, total_size):
        if total_size <= 0:
            return
        downloaded = min(block_count * block_size, total_size)
        percent = downloaded / total_size * 100
        print(
            f"\r{filename}: {percent:6.2f}% "
            f"({downloaded / 1024 / 1024:.2f}/{total_size / 1024 / 1024:.2f} MB)",
            end="",
        )
    return report

for filename, url in files.items():
    target = DATA_DIR / filename
    if target.exists() and target.stat().st_size > 0:
        print(f"{filename}: already exists ({target.stat().st_size / 1024 / 1024:.2f} MB)")
        continue
    print(f"Downloading {filename}...")
    urlretrieve(url, target, show_progress(filename))
    print(f"\nSaved to {target}")

train.txt: 100.00% (1.08/1.08 MB)
Saved to d:\Code\unsia\enexre\data\bc5cdr\train.txt
dev.txt: 100.00% (1.08/1.08 MB)
Saved to d:\Code\unsia\enexre\data\bc5cdr\dev.txt
test.txt: 100.00% (1.11/1.11 MB)
Saved to d:\Code\unsia\enexre\data\bc5cdr\test.txt


In [17]:
for filename in ["train.txt", "dev.txt", "test.txt"]:
    path = DATA_DIR / filename
    if not path.exists():
        print(f"{filename} -> missing")
        continue
    print(f"{filename} -> {path.stat().st_size / 1024 / 1024:.2f} MB")

train.txt -> 1.08 MB
dev.txt -> 1.08 MB
test.txt -> 1.11 MB


In [18]:
with (DATA_DIR / "train.txt").open(encoding="utf-8") as file:
    for _ in range(20):
        print(file.readline().rstrip())

227508|t|Naloxone reverses the antihypertensive effect of clonidine.
227508|a|In unanesthetized, spontaneously hypertensive rats the decrease in blood pressure and heart rate produced by intravenous clonidine, 5 to 20 micrograms/kg, was inhibited or reversed by nalozone, 0.2 to 2 mg/kg. The hypotensive effect of 100 mg/kg alpha-methyldopa was also partially reversed by naloxone. Naloxone alone did not affect either blood pressure or heart rate. In brain membranes from spontaneously hypertensive rats clonidine, 10(-8) to 10(-5) M, did not influence stereoselective binding of [3H]-naloxone (8 nM), and naloxone, 10(-8) to 10(-4) M, did not influence clonidine-suppressible binding of [3H]-dihydroergocryptine (1 nM). These findings indicate that in spontaneously hypertensive rats the effects of central alpha-adrenoceptor stimulation involve activation of opiate receptors. As naloxone and clonidine do not appear to interact with the same receptor site, the observed functional antagonism sugg